## Build a Simple LLM Application with LCEL:

This application will translate text from English to another language.This is relatively simple LLM application - it's just a single LLM call plussome prompting.

- LCEL: LangChain Expression Language.
- Langserve

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key=os.getenv("GROQ_API_KEY")


In [2]:
from langchain_groq import ChatGroq
model=ChatGroq(model="llama-3.3-70b-versatile",groq_api_key=groq_api_key)
model

d:\CODES\Gen AI course\envi\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ChatGroq(output_version=None, profile={'name': 'Llama 3.3 70B Versatile', 'release_date': '2024-12-06', 'last_updated': '2024-12-06', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000002342BAC88B0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002342BAC8FA0>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [3]:
from langchain_core.messages import HumanMessage,SystemMessage
messages=[
    SystemMessage(content="translate from English to French"),
    HumanMessage(content="Hello How are you?")
]

result=model.invoke(messages)

In [4]:
result

AIMessage(content='Bonjour, comment allez-vous ? \n\n(Translation: Hello, how are you?)', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 45, 'total_tokens': 63, 'completion_time': 0.067053183, 'completion_tokens_details': None, 'prompt_time': 0.007385115, 'prompt_tokens_details': None, 'queue_time': 0.315904252, 'total_time': 0.074438298}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_f8b414701e', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ec0ab-3458-7841-a7b5-1d8d2ec5cd41-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 45, 'output_tokens': 18, 'total_tokens': 63})

In [5]:
from langchain_core.output_parsers import StrOutputParser
output_parser=StrOutputParser()
output_parser.invoke(result)

'Bonjour, comment allez-vous ? \n\n(Translation: Hello, how are you?)'

In [6]:
## using LCEL chaining the components:
chain=model|output_parser
chain.invoke(messages)

'Bonjour, comment allez-vous ? \n\n(Note: I translated "Hello, how are you?" into French. If you\'d like, I can also respond with a typical French greeting, such as "Bonjour, je vais bien, merci !")'

In [7]:
## Prompt templates:
from langchain_core.prompts import ChatPromptTemplate

generic_template="translate the following data into {language}:"
prompt=ChatPromptTemplate.from_messages(
    [
        ("system",generic_template),
        ("user","{text}")
    ]
)


In [8]:

result=prompt.invoke({"language":"French","text":"Hello"})
result.to_messages()

[SystemMessage(content='translate the following data into French:', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Hello', additional_kwargs={}, response_metadata={})]

In [9]:
chain=prompt|model|output_parser
chain.invoke({"language":"French","text":"Hello"})

"Bonjour ! Pour traduire les données, j'aurai besoin que vous me fournissiez le texte à traduire. Pouvez-vous me donner les informations que vous souhaitez voir traduites en français ?"